In [3]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier


In [13]:
import psutil

mem = psutil.virtual_memory()
print(f"Total RAM: {mem.total/1e9:.2f} GB")
print(f"Available: {mem.available/1e9:.2f} GB")
print(f"Used: {mem.used/1e9:.2f} GB")

Total RAM: 33.65 GB
Available: 22.36 GB
Used: 10.80 GB


In [10]:
import os
import psutil

process = psutil.Process(os.getpid())
print(f"RSS: {process.memory_info().rss/1024**3:.2f} GB")

RSS: 17.76 GB


In [4]:
## Load the data
adata = sc.read_h5ad("../data/raw/rna.h5ad")

In [11]:
# create a random subset of adata
subset_indices = np.random.choice(adata.shape[0], size=int(0.1 * adata.shape[0]), replace=False)
subset_adata = adata[subset_indices]

In [12]:
X_data = subset_adata.X.toarray()

In [14]:
## Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_data, subset_adata.obs['perturbation_2'], test_size=0.2, random_state=42)

In [20]:
classes = subset_adata.obs['perturbation_2'].unique().tolist()

In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

mlp_clf = MLPClassifier(hidden_layer_sizes=[300, 100], verbose=True, early_stopping=True, random_state=42)
pipeline = make_pipeline(StandardScaler(), mlp_clf)
pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

print(f"Model Accuracy: {accuracy:.4f}")

Iteration 1, loss = 0.24066446
Validation score: 0.959931
Iteration 2, loss = 0.02272391
Validation score: 0.967373
Iteration 3, loss = 0.03181854
Validation score: 0.958786
Iteration 4, loss = 0.00679185
Validation score: 0.966228
Iteration 5, loss = 0.00129224
Validation score: 0.977104
Iteration 6, loss = 0.00035482
Validation score: 0.978248
Iteration 7, loss = 0.00034151
Validation score: 0.977676
Iteration 8, loss = 0.00033454
Validation score: 0.978248
Iteration 9, loss = 0.00033158
Validation score: 0.977104
Iteration 10, loss = 0.00032677
Validation score: 0.978248
Iteration 11, loss = 0.00032443
Validation score: 0.977676
Iteration 12, loss = 0.00032142
Validation score: 0.978248
Iteration 13, loss = 0.00031952
Validation score: 0.978248
Iteration 14, loss = 0.00031780
Validation score: 0.978248
Iteration 15, loss = 0.00031564
Validation score: 0.977676
Iteration 16, loss = 0.00031454
Validation score: 0.978248
Iteration 17, loss = 0.00031262
Validation score: 0.978248
Valida

# pytorch version

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

ModuleNotFoundError: No module named 'torchmetrics'

In [17]:
class ConditionClassifierMLP(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )
        
    def forward(self, X):
        return self.mlp(X)

In [18]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
model = ConditionClassifierMLP(n_inputs=X_train.shape[1], n_hidden1=300, n_hidden2=100, n_classes=len(classes)).to(device)
xentropy = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [24]:
class_to_idx = {name: idx for idx, name in enumerate(classes)}
y_train_idx = [class_to_idx[label] for name, label in y_train.items()]
y_test_idx = [class_to_idx[label] for name, label in y_test.items()]

y_train_tensor = torch.tensor(y_train_idx, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_idx, dtype=torch.long)

if hasattr(X_train, "toarray"):
    X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test.toarray(), dtype=torch.float32)
else:
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

In [25]:
from torch.utils.data import TensorDataset, DataLoader

# Combine X and y into a standard unified PyTorch Dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders for chunking data into manageable mini-batches
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [26]:
# We track loss values over time to monitor training convergence
loss_history = []
EPOCHS = 20

model.train()  # Set the model to training mode
for epoch in range(EPOCHS):
    running_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        # Move mini-batch tensors directly onto your hardware device (CPU/GPU)
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # 1. Clear out stale gradients from the previous iteration step
        optimizer.zero_grad()
        
        # 2. Run the forward execution pass to generate class predictions
        predictions = model(batch_X)
        
        # 3. Measure prediction performance error using Cross Entropy
        loss = xentropy(predictions, batch_y)
        
        # 4. Perform backpropagation to calculate error gradients
        loss.backward()
        
        # 5. Tweak internal model weights using the Adam optimizer
        optimizer.step()
        
        running_loss += loss.item() * batch_X.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    loss_history.append(epoch_loss)
    print(f"Epoch [{epoch+1}/{EPOCHS}] -> Training Loss: {epoch_loss:.4f}")

Epoch [1/20] -> Training Loss: 0.1668
Epoch [2/20] -> Training Loss: 0.0406
Epoch [3/20] -> Training Loss: 0.0275
Epoch [4/20] -> Training Loss: 0.0140
Epoch [5/20] -> Training Loss: 0.0163
Epoch [6/20] -> Training Loss: 0.0148
Epoch [7/20] -> Training Loss: 0.0135
Epoch [8/20] -> Training Loss: 0.0181
Epoch [9/20] -> Training Loss: 0.0046
Epoch [10/20] -> Training Loss: 0.0020
Epoch [11/20] -> Training Loss: 0.0002
Epoch [12/20] -> Training Loss: 0.0001
Epoch [13/20] -> Training Loss: 0.0001
Epoch [14/20] -> Training Loss: 0.0000
Epoch [15/20] -> Training Loss: 0.0000
Epoch [16/20] -> Training Loss: 0.0000
Epoch [17/20] -> Training Loss: 0.0000
Epoch [18/20] -> Training Loss: 0.0000
Epoch [19/20] -> Training Loss: 0.0000
Epoch [20/20] -> Training Loss: 0.0000


In [27]:
model.eval()  # Set the model to evaluation mode
correct_predictions = 0
total_samples = 0

# Deactivate gradient tracking memory allocation since we are only calculating predictions
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass execution
        outputs = model(batch_X)
        
        # Determine the prediction index with the highest logit score probability
        _, predicted_classes = torch.max(outputs, dim=1)
        
        # Accumulate metrics
        total_samples += batch_y.size(0)
        correct_predictions += (predicted_classes == batch_y).sum().item()

# Print finalized performance metric evaluation
final_accuracy = (correct_predictions / total_samples) * 100
print(f"\nEvaluation Complete!")
print(f"Final Validation Accuracy: {final_accuracy:.2f}%")


Evaluation Complete!
Final Validation Accuracy: 98.86%
